In [54]:
from typing import List, Tuple

from datasets import load_dataset, Dataset, concatenate_datasets
import evaluate
import numpy as np
from sklearn.metrics import classification_report
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments, Trainer
from transformers import pipeline
import random

SEED = 42

In [55]:
MODEL_NAME = 'DeepPavlov/rubert-base-cased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [56]:
seq = 'Привет о дивный новый мир!'
print(tokenizer.encode(seq))
print(tokenizer.convert_ids_to_tokens(tokenizer.encode(seq)))

[101, 77527, 612, 32136, 1916, 10303, 6913, 106, 102]
['[CLS]', 'Привет', 'о', 'див', '##ный', 'новый', 'мир', '!', '[SEP]']


In [57]:
DATASET_NAME = 'Davlan/sib200'
DATASET_LANGUAGE = 'rus_Cyrl'
train_set = load_dataset(DATASET_NAME, DATASET_LANGUAGE, split='train')
validation_set = load_dataset(DATASET_NAME, DATASET_LANGUAGE, split='validation')
test_set = load_dataset(DATASET_NAME, DATASET_LANGUAGE, split='test')

print(train_set)

Dataset({
    features: ['index_id', 'category', 'text'],
    num_rows: 701
})


In [58]:
def augment_text(text, p_swap=0.1):
    words = text.split()
    for i in range(len(words)-1):
        if random.random() < p_swap:
            words[i], words[i+1] = words[i+1], words[i]
    return ' '.join(words)

# создаем аугментированные тексты
aug_texts = [augment_text(t, p_swap=0.1) for t in train_set['text']]
aug_labels = train_set['category']

# новый датасет с аугментацией
aug_dataset = Dataset.from_dict({
    'text': aug_texts,
    'category': aug_labels
})

# объединяем с оригиналом
combined_train_set = concatenate_datasets([train_set, aug_dataset])
print(f"Original size: {len(train_set)}, after augmentation: {len(combined_train_set)}")

Original size: 701, after augmentation: 1402


In [78]:
MINIBATCH_SIZE = 64
tokenized_train_set = combined_train_set.map(
    lambda it: tokenizer(it['text'], truncation=True),
    batched=True, batch_size=MINIBATCH_SIZE
)
tokenized_validation_set = validation_set.map(
    lambda it: tokenizer(it['text'], truncation=True),
    batched=True, batch_size=MINIBATCH_SIZE
)

print(tokenized_train_set)

Map: 100%|██████████| 1402/1402 [00:00<00:00, 11770.69 examples/s]

Dataset({
    features: ['index_id', 'category', 'text', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1402
})


In [79]:
cls_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return cls_metric.compute(predictions=predictions, references=labels, average='macro')

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [80]:
list_of_categories = sorted(list(
    set(combined_train_set['category']) |
    set(validation_set['category']) |
    set(test_set['category'])
))
indices_of_categories = list(range(len(list_of_categories)))
n_categories = len(list_of_categories)
print(f'Categories for classification are: {list_of_categories}')
id2label = dict(zip(indices_of_categories, list_of_categories))
label2id = dict(zip(list_of_categories, indices_of_categories))

labeled_train_set = tokenized_train_set.add_column(
    'label',
    [label2id[val] for val in tokenized_train_set['category']]
)
labeled_validation_set = tokenized_validation_set.add_column(
    'label',
    [label2id[val] for val in tokenized_validation_set['category']]
)


Categories for classification are: ['entertainment', 'geography', 'health', 'politics', 'science/technology', 'sports', 'travel']


In [81]:
classifier = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=n_categories,
    id2label=id2label,
    label2id=label2id
).cuda()

for param in classifier.parameters():
    param.data = param.data.contiguous()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [82]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 4070 Ti SUPER


In [83]:
# training_args = TrainingArguments(
#     output_dir='rubert_sib200',
#     learning_rate=2e-5,
#     per_device_train_batch_size=MINIBATCH_SIZE,
#     per_device_eval_batch_size=MINIBATCH_SIZE,
#     num_train_epochs=20,
#     weight_decay=1e-3,
#     eval_strategy='epoch',
#     save_strategy='epoch',
#     load_best_model_at_end=True,
#     metric_for_best_model='eval_loss',
#     logging_steps=5,
#     data_seed=SEED,
#     bf16=True,           # <<< Включить GPU-ускорение
#     gradient_accumulation_steps=1,
# )

In [84]:
training_args = TrainingArguments(
    output_dir='rubert_sib200_aug',
    overwrite_output_dir=True,         # перезаписывать старые модели
    learning_rate=1.5e-5,                # чуть выше, чтобы быстрее сходилось
    per_device_train_batch_size=32,    # увеличиваем, если память GPU позволяет
    per_device_eval_batch_size=64,     # для более стабильной оценки
    num_train_epochs=50,               # меньше, если используем LR scheduler
    weight_decay=0.01,                 # небольшая регуляризация
    metric_for_best_model='f1',        # ориентируемся на f1 макро
    greater_is_better=True,
    logging_steps=20,
    bf16=True,                         # GPU ускорение
    gradient_accumulation_steps=2,     # увеличивает effective batch size
    lr_scheduler_type='cosine',        # плавное уменьшение LR
    warmup_ratio=0.1,                  # 10% шагов — warmup
    dataloader_num_workers=8,
    run_name='rubert_sib200_f1_optim',
    report_to='none',                  # отключаем wandb/трекинг, если не используем
    eval_strategy='epoch',       # оценка каждый epoch
    save_strategy='epoch',       # сохраняем при каждой оценке
    save_total_limit=1,          # оставляем только лучшую модель
    load_best_model_at_end=True, # после тренировки вернётся лучшая
)


In [85]:
trainer = Trainer(
    model=classifier,
    args=training_args,
    train_dataset=labeled_train_set,
    eval_dataset=labeled_validation_set,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

/tmp/ipykernel_977/1253556369.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [86]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,1.960800,1.933318,0.096575
2,1.903300,1.816840,0.182586
3,1.776700,1.619239,0.261653
4,1.556800,1.282710,0.452692
5,1.224000,0.769719,0.847387
6,0.698300,0.510767,0.825565
7,0.262800,0.597384,0.830764
8,0.104300,0.650444,0.825328
9,0.038400,0.753150,0.825328
10,0.020500,0.821953,0.825328


TrainOutput(global_step=1100, training_loss=0.17664950162849644, metrics={'train_runtime': 373.3857, 'train_samples_per_second': 187.742, 'train_steps_per_second': 2.946, 'total_flos': 1929680410799100.0, 'train_loss': 0.17664950162849644, 'epoch': 50.0})

In [91]:
trainer.evaluate()

{'eval_loss': 0.8473389744758606,
 'eval_f1': 0.8495995095793336,
 'eval_runtime': 0.2659,
 'eval_samples_per_second': 372.282,
 'eval_steps_per_second': 7.521,
 'epoch': 50.0}

In [92]:
classifiсation_pipeline = pipeline('text-classification', model=classifier, tokenizer=tokenizer, device=0)

Device set to use cuda:0


In [93]:
y_pred = list(map(lambda x: x['label'], classifiсation_pipeline(list(validation_set['text']))))
y_true = validation_set['category']
print(classification_report(y_true=y_true, y_pred=y_pred))

                    precision    recall  f1-score   support

     entertainment       0.86      0.67      0.75         9
         geography       0.78      0.88      0.82         8
            health       1.00      0.82      0.90        11
          politics       0.87      0.93      0.90        14
science/technology       0.96      0.92      0.94        25
            sports       1.00      0.92      0.96        12
            travel       0.62      0.75      0.68        20

          accuracy                           0.85        99
         macro avg       0.87      0.84      0.85        99
      weighted avg       0.86      0.85      0.85        99



In [94]:
y_pred = list(map(lambda x: x['label'], classifiсation_pipeline(list(test_set['text']))))
y_true = test_set['category']
print(classification_report(y_true=y_true, y_pred=y_pred))

                    precision    recall  f1-score   support

     entertainment       0.85      0.58      0.69        19
         geography       0.93      0.82      0.88        17
            health       1.00      0.86      0.93        22
          politics       0.93      0.93      0.93        30
science/technology       0.86      0.94      0.90        51
            sports       0.92      0.96      0.94        25
            travel       0.87      0.97      0.92        40

          accuracy                           0.90       204
         macro avg       0.91      0.87      0.88       204
      weighted avg       0.90      0.90      0.89       204

